In [1]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [2]:

load_dotenv()

True

In [3]:
class ComplaintData(BaseModel):
    complaint: str = Field(
        description="The main problem reported by the customer"
    )
    product_or_feature: str = Field(
        description="The product or feature related to the complaint"
    )
    sentiment: str = Field(
        description="Customer sentiment: Positive, Negative, or Neutral"
    )


In [4]:
# Create the Pydantic output parser
parser = PydanticOutputParser(
    pydantic_object=ComplaintData
)


# Initialize the NVIDIA model
model = ChatNVIDIA(
    model="nvidia/nemotron-3-super-120b-a12b",
    temperature=0
)

In [5]:
# Step 1: Create the prompt template
step1_prompt = PromptTemplate(
    template="""
You are a customer review analyzer.

Extract these three fields from the review:
1. complaint: the core customer complaint
2. product_or_feature: the product or feature involved
3. sentiment: Positive, Negative, or Neutral

Use only information supported by the review.
If the product or feature is not mentioned, write "Not specified".

Customer review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

In [11]:

# Connect Step 1
step1_chain = step1_prompt | model | parser


# Step 2: Support ticket writer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

step2_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a support ticket writer.
        Create a concise, professional support ticket summary
        for the internal customer support team.

        Use ONLY the structured complaint data provided.
        Do not ask for or invent information that is not provided.
        Include:
        - Complaint
        - Product or feature
        - Customer sentiment
        - Recommended next action based on the complaint
        """
    ),
    (
        "human",
        """Structured complaint data:
        Complaint: {complaint}
        Product or feature: {product_or_feature}
        Sentiment: {sentiment}
        """
    )
])

# Parse Step 2 as plain text
step2_chain = step2_prompt | model | StrOutputParser()


# Run the complete two-step pipeline
if __name__ == "__main__":
    review = (
        "The mobile app crashes every time I try to make a payment. "
        "I am very frustrated."
    )

    # Step 1: Raw review -> structured data
    structured_data = step1_chain.invoke(
        {"review_text": review}
    )

    print("\nSTEP 1: STRUCTURED COMPLAINT DATA")
    print(structured_data.model_dump_json(indent=2))

    # Step 2: Structured data only -> support ticket
    ticket_summary = step2_chain.invoke(
        {
            "complaint": structured_data.complaint,
            "product_or_feature": structured_data.product_or_feature,
            "sentiment": structured_data.sentiment,
        }
    )

    print("\nSTEP 2: SUPPORT TICKET SUMMARY")
    print(ticket_summary)





STEP 1: STRUCTURED COMPLAINT DATA
{
  "complaint": "mobile app crashes every time I try to make a payment",
  "product_or_feature": "mobile app",
  "sentiment": "Negative"
}

STEP 2: SUPPORT TICKET SUMMARY
**Support Ticket Summary**

- **Complaint:** Mobile app crashes every time a payment is attempted.  
- **Product/Feature:** Mobile app (payment functionality).  
- **Customer Sentiment:** Negative.  
- **Recommended Next Action:** Reproduce the crash on a test device, collect device logs and error reports, identify the root cause in the payment flow, and prioritize a fix or patch release.


## Test with a different customer review

In [12]:
if __name__ == "__main__":
    review = (
        "The headphones have excellent sound quality, "
        "but the battery drains too quickly."
    )

    # Step 1: Raw review -> structured data
    structured_data = step1_chain.invoke(
        {"review_text": review}
    )

    print("\nSTEP 1: STRUCTURED COMPLAINT DATA")
    print(structured_data.model_dump_json(indent=2))


STEP 1: STRUCTURED COMPLAINT DATA
{
  "complaint": "battery drains too quickly",
  "product_or_feature": "headphones",
  "sentiment": "Neutral"
}
